# Fine-Tune MDR — Resume Training on Colab

**Model:** `fdtn-ai/Foundation-Sec-8B-Instruct` (LoRA fine-tune)

Resumes training from a DGX checkpoint on Google Drive.
Saves checkpoints to **local disk** (fast, reliable), and a **DriveBackupCallback**
copies each checkpoint to Drive after it's written. If Drive fails, training keeps going.

## Prerequisites

1. Upload checkpoint folder to `My Drive/fine-tune-mdr/outputs/checkpoint-NNNN/`
2. Add HuggingFace token to Colab Secrets (key: `HF_TOKEN`)
3. Select a **high-VRAM GPU** runtime (A100 80GB or H100)

In [ ]:
# Cell 1: GPU Verification
import torch

assert torch.cuda.is_available(), "No GPU detected! Change runtime to GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

if vram_gb < 39:
    print(f"\nWARNING: Only {vram_gb:.0f} GB VRAM. Batch size 16 will OOM.")
    print("Change BATCH_SIZE to 2 and GRAD_ACCUM to 16 in the config cell.")

In [ ]:
# Cell 2: Install Dependencies
!pip install -q \
    transformers>=4.57 \
    peft>=0.18 \
    trl>=0.23 \
    datasets \
    accelerate \
    pyyaml \
    tensorboard

In [ ]:
# Cell 3: Mount Google Drive + Set Paths
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/fine-tune-mdr"
DRIVE_OUTPUT = f"{DRIVE_BASE}/outputs"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Training saves to LOCAL disk (fast, reliable)
# DriveBackupCallback copies to Drive after each save
OUTPUT_DIR = "/content/outputs"
LOGGING_DIR = "/content/logs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

print(f"Local output: {OUTPUT_DIR}")
print(f"Drive backup: {DRIVE_OUTPUT}")

In [ ]:
# Cell 4: HuggingFace Token
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    if "HF_TOKEN" not in os.environ:
        raise RuntimeError(
            "HF_TOKEN not found. Add it to Colab Secrets "
            "(key icon in left sidebar) with key 'HF_TOKEN'."
        )

In [ ]:
# Cell 5: Find Latest Checkpoint + Clean Up Empty Folders
import json
import shutil

def find_latest_valid_checkpoint(*search_dirs):
    """Find the highest-numbered VALID checkpoint across search dirs.
    Skips empty/incomplete checkpoints and deletes them."""
    required = ["adapter_model.safetensors", "trainer_state.json", "optimizer.pt", "scheduler.pt"]
    best = None
    best_step = -1
    for d in search_dirs:
        p = Path(d)
        if not p.exists():
            continue
        for ckpt in p.glob("checkpoint-*"):
            try:
                step = int(ckpt.name.split("-")[1])
            except (IndexError, ValueError):
                continue
            # Check if checkpoint is complete
            missing = [f for f in required if not (ckpt / f).exists()]
            if missing:
                print(f"  Skipping {ckpt.name} (missing {len(missing)} files)")
                try:
                    shutil.rmtree(ckpt)
                    print(f"  Deleted empty {ckpt.name}")
                except Exception:
                    pass
                continue
            if step > best_step:
                best_step = step
                best = str(ckpt)
    return best

print("Searching for valid checkpoints...")
RESUME_CHECKPOINT = find_latest_valid_checkpoint(OUTPUT_DIR, DRIVE_OUTPUT)

if RESUME_CHECKPOINT:
    ckpt_path = Path(RESUME_CHECKPOINT)
    with open(ckpt_path / "trainer_state.json") as f:
        ckpt_state = json.load(f)
    last_log = ckpt_state.get("log_history", [{}])[-1]
    print(f"\nFound valid checkpoint: {RESUME_CHECKPOINT}")
    print(f"  Step:  {ckpt_state['global_step']}")
    print(f"  Loss:  {last_log.get('loss', 'N/A')}")
    print(f"  LR:    {last_log.get('learning_rate', 'N/A')}")
else:
    print("\nNo valid checkpoint found. Will start fresh.")
    print(f"Searched: {OUTPUT_DIR}, {DRIVE_OUTPUT}")

In [ ]:
# Cell 6: Configuration

# --- Model ---
MODEL_NAME = "fdtn-ai/Foundation-Sec-8B-Instruct"
MAX_SEQ_LENGTH = 1024
ATTN_IMPLEMENTATION = "sdpa"  # Built into PyTorch, no flash-attn install needed

# --- LoRA (must match checkpoint exactly) ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LORA_BIAS = "none"
USE_RSLORA = False

# --- Dataset ---
DATASET_NAME = "jason-oneal/mitre-stix-cve-exploitdb-dataset-alpaca-chatml-harmony"
DATASET_CONFIG = "chatml"

# --- Training ---
BATCH_SIZE = 16           # For 80GB+ VRAM. Use 2 for 40GB.
GRAD_ACCUM = 2            # Effective batch = 16*2 = 32
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 10
LR_SCHEDULER = "cosine"
OPTIMIZER = "adamw_torch"
MAX_STEPS = 17479
SAVE_STEPS = 500          # Save every ~25 min at 3 sec/step
SAVE_TOTAL_LIMIT = 3
LOGGING_STEPS = 10
SEED = 42
TIME_LIMIT_HOURS = 23

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Max steps: {MAX_STEPS}")
if RESUME_CHECKPOINT:
    remaining = MAX_STEPS - ckpt_state['global_step']
    print(f"Remaining steps: {remaining}")

In [ ]:
# Cell 7: CUDA Setup
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("TF32 enabled, expandable_segments enabled.")

In [ ]:
# Cell 8: Load Tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab size: {len(tokenizer):,}")

In [ ]:
# Cell 9: Load Dataset + Apply Chat Template
from datasets import load_dataset

print(f"Loading dataset: {DATASET_NAME}...")
raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
print(f"Loaded {len(raw_ds):,} examples.")

columns = raw_ds.column_names
if "text" not in columns:
    print("Applying chat template...")
    train_ds = raw_ds.map(
        lambda ex: {"text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )},
        remove_columns=[c for c in columns if c != "text"],
        desc="Formatting",
    )
else:
    train_ds = raw_ds

print(f"Training examples: {len(train_ds):,}")

In [ ]:
# Cell 10: Clear HuggingFace Cache (free disk space)
!rm -rf /root/.cache/huggingface/hub /root/.cache/huggingface/datasets
!df -h /content
print("HF cache cleared.")

In [ ]:
# Cell 11: Load Base Model
from transformers import AutoModelForCausalLM

print(f"Loading {MODEL_NAME} in BF16 with {ATTN_IMPLEMENTATION}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPLEMENTATION,
    device_map={"": 0},
)
print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 12: Apply LoRA (must match checkpoint config exactly)
from peft import LoraConfig, get_peft_model, TaskType

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias=LORA_BIAS,
    task_type=TaskType.CAUSAL_LM,
    use_rslora=USE_RSLORA,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 13: Callbacks (TimeLimitCallback + DriveBackupCallback)
import time
import shutil
import threading
from transformers import TrainerCallback

class TimeLimitCallback(TrainerCallback):
    """Stops training before Colab's 24h limit."""
    def __init__(self, time_limit_hours):
        self.time_limit_seconds = time_limit_hours * 3600
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"TimeLimitCallback: will auto-save after {self.time_limit_seconds / 3600:.0f}h")

    def on_step_end(self, args, state, control, **kwargs):
        if self.start_time is None:
            return
        elapsed = time.time() - self.start_time
        if elapsed >= self.time_limit_seconds:
            print(f"\nTimeLimitCallback: {elapsed / 3600:.1f}h elapsed, stopping...")
            control.should_training_stop = True
            control.should_save = True


class DriveBackupCallback(TrainerCallback):
    """Copies checkpoints from local disk to Google Drive after each save.
    Runs in a background thread so it never blocks training.
    If Drive fails, training keeps going."""

    def __init__(self, local_dir, drive_dir, keep=2):
        self.local_dir = Path(local_dir)
        self.drive_dir = Path(drive_dir)
        self.keep = keep  # number of checkpoints to keep on Drive
        self.drive_dir.mkdir(parents=True, exist_ok=True)

    def _backup(self, ckpt_name):
        """Copy one checkpoint to Drive and prune old ones."""
        src = self.local_dir / ckpt_name
        dst = self.drive_dir / ckpt_name
        try:
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            # Verify copy
            src_files = set(f.name for f in src.iterdir())
            dst_files = set(f.name for f in dst.iterdir())
            if src_files == dst_files:
                print(f"  [DriveBackup] {ckpt_name} copied to Drive ({len(dst_files)} files)")
            else:
                print(f"  [DriveBackup] WARNING: {ckpt_name} copy incomplete!")
            # Prune old checkpoints on Drive
            drive_ckpts = sorted(self.drive_dir.glob("checkpoint-*"),
                                 key=lambda p: int(p.name.split('-')[1]))
            while len(drive_ckpts) > self.keep:
                old = drive_ckpts.pop(0)
                shutil.rmtree(old)
                print(f"  [DriveBackup] Pruned old {old.name}")
        except Exception as e:
            print(f"  [DriveBackup] WARNING: backup failed ({e}), training continues")

    def on_save(self, args, state, control, **kwargs):
        """Called after trainer saves a checkpoint."""
        ckpt_name = f"checkpoint-{state.global_step}"
        if (self.local_dir / ckpt_name).exists():
            # Run backup in background thread so it doesn't block training
            t = threading.Thread(target=self._backup, args=(ckpt_name,), daemon=True)
            t.start()

print("Callbacks ready.")

In [ ]:
# Cell 14: Configure Trainer
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type=LR_SCHEDULER,
    optim=OPTIMIZER,
    max_steps=MAX_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    fp16=False,
    bf16=True,
    seed=SEED,
    report_to="tensorboard",
    logging_dir=LOGGING_DIR,
    eval_strategy="no",
    dataloader_num_workers=4,
    torch_compile=False,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=sft_config,
    callbacks=[
        TimeLimitCallback(TIME_LIMIT_HOURS),
        DriveBackupCallback(OUTPUT_DIR, DRIVE_OUTPUT, keep=2),
    ],
)

print(f"Trainer ready.")
print(f"Local output: {OUTPUT_DIR}")
print(f"Drive backup: {DRIVE_OUTPUT}")
print(f"Resume: {RESUME_CHECKPOINT or 'None (fresh start)'}")

In [ ]:
# Cell 15: Train!
print("Starting training...")
if RESUME_CHECKPOINT:
    print(f"Resuming from {Path(RESUME_CHECKPOINT).name}")

trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)

print(f"\nTraining complete!")
print(f"Final step: {trainer.state.global_step} / {trainer.state.max_steps}")

In [ ]:
# Cell 16: Save Final Adapter to Drive
import gc

FINAL_DIR = f"{DRIVE_OUTPUT}/final_adapter"
os.makedirs(FINAL_DIR, exist_ok=True)

print(f"Saving final LoRA adapter to {FINAL_DIR}...")
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Done! Final adapter saved to Google Drive.")

# Verify
saved_files = list(Path(FINAL_DIR).iterdir())
print(f"Files saved: {[f.name for f in saved_files]}")

# Free memory
del trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 17: Quick Inference Test
print("Running quick inference test...")

test_prompt = [
    {"role": "system", "content": "You are a cybersecurity threat detection assistant."},
    {"role": "user", "content": (
        "Analyze this log entry for threats:\n"
        "EventID=4625 LogonType=10 TargetUserName=Administrator "
        "IpAddress=185.220.101.34 WorkstationName=UNKNOWN "
        "FailureReason=%%2313 SubStatus=0xC000006D"
    )},
]

inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        inputs, max_new_tokens=256, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"\n--- Model Response ---\n{response}")

## Download Results to DGX

### From shared drive
```bash
# Download final_adapter from Google Drive to shared mount
cp -r /mnt/ai_projects/checkpoints/final_adapter ~/Documents/dgx-code-bank/fine-tune-mdr/outputs/final_adapter
```

### Or direct from Drive
1. Go to [Google Drive](https://drive.google.com)
2. Download `My Drive/fine-tune-mdr/outputs/final_adapter/`
3. `scp -r final_adapter/ jayoung@dgx:~/Documents/dgx-code-bank/fine-tune-mdr/outputs/`

### Then run eval
```bash
cd ~/Documents/dgx-code-bank/fine-tune-mdr
python eval.py
```

### If Colab timed out
Download the latest `checkpoint-NNNN/` from Drive `outputs/`, copy to DGX `outputs/`, then:
```bash
python train_native.py  # auto-detects latest checkpoint
```